In [ ]:
from pathlib import Path
import json
import random

In [ ]:
def get_project_root() -> Path:
    current = Path.cwd()
    for candidate in [current, *current.parents]:
        if (candidate / "data_preprocess").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Unable to locate project root.")

PROJECT_ROOT = get_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / "data_mixing"

Total tokens for the sampled 323000 items: 174039959


In [ ]:
experiment_name = "exp2"
model_name = "Llama-3.1-8B"
base_token = 200_000_000

token_limits = {
    f"{base_token}_{model_name}_instr_optim": int(base_token * 0.48666725),
    f"{base_token}_{model_name}_math_optim": int(base_token * 0.29281993),
    f"{base_token}_{model_name}_code_optim": int(base_token * 0.22051281),
}

DOMAIN_DATASETS = {
    "code": DATA_DIR / "opencoder-sft_len.json",
    "instr": DATA_DIR / "Infinity-Instruct_0625_len.json",
    "math": DATA_DIR / "openmathinstruct2_1M_len.json",
}


def sample_tokens(data, token_limit):
    selected = []
    total_tokens = 0
    for item in data:
        if total_tokens + item["len"] <= token_limit:
            total_tokens += item["len"]
            selected.append({k: v for k, v in item.items() if k != "len"})
        else:
            remaining_tokens = token_limit - total_tokens
            truncated_item = {k: v for k, v in item.items() if k != "len"}
            truncated_item["output"] = truncated_item.get("output", "")[:remaining_tokens]
            selected.append(truncated_item)
            break
    return selected

## Start sample training data

In [ ]:
experiment_dir = OUTPUT_DIR / experiment_name
experiment_dir.mkdir(parents=True, exist_ok=True)

for domain, dataset_path in DOMAIN_DATASETS.items():
    with dataset_path.open("r") as f:
        structured_data = json.load(f)

    random.shuffle(structured_data)

    for name, limit in token_limits.items():
        if domain not in name:
            continue

        sampled_items = sample_tokens(structured_data, limit)
        subset_path = experiment_dir / f"{base_token}_{domain}_{name}.json"
        with subset_path.open("w") as f:
            json.dump(sampled_items, f, indent=2)

        print(f"Saved {subset_path.name} with {len(sampled_items)} items")

Sampled 95702 items for 200000000_Llama-3.1-8B_code_optim with token limit 44102562.


Sampled 180460 items for 200000000_Llama-3.1-8B_instr_optim with token limit 97333450.


Sampled 132255 items for 200000000_Llama-3.1-8B_math_optim with token limit 58563986.


## Store data name into data_info

In [ ]:
dataset_info_path = DATA_DIR / "dataset_info.json"

if dataset_info_path.exists():
    with dataset_info_path.open("r") as f:
        try:
            dataset_info = json.load(f)
        except json.JSONDecodeError:
            dataset_info = {}
else:
    dataset_info = {}

for domain in DOMAIN_DATASETS:
    for name in token_limits:
        if domain not in name:
            continue

        dataset_name = f"{base_token}_{domain}_{name}"
        relative_output = Path("data_mixing") / experiment_name / f"{base_token}_{domain}_{name}.json"
        dataset_info[dataset_name] = {"file_name": str(relative_output)}

with dataset_info_path.open("w") as f:
    json.dump(dataset_info, f, indent=2)

200000000_instr_200000000_Llama-3.1-8B_instr_optim
200000000_math_200000000_Llama-3.1-8B_math_optim
200000000_code_200000000_Llama-3.1-8B_code_optim
